# Week 2, Option B3: 5-fold cross-validation

MSE loss, constant lr 1e-4, 50 epochs, seed 42. KFold(5, shuffle=True, random_state=42) over all 1000 images. Each fold trains on 800 and evaluates on 200 images (no separate test set). The model after the last epoch is evaluated. Requires a GPU runtime and `LDCTIQAG2023_train.zip` in /content.

In [ ]:
!git clone https://github.com/neuronautsai/ldct-iqac-resnet18.git
%cd ldct-iqac-resnet18
!git checkout 90afae67d1fea3ac10edecd43fde0360cf3fb4f1
!pip install -q imageio pandas matplotlib scipy scikit-learn tqdm

In [ ]:
import os, glob, zipfile, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from config import Config
from data import load_master_list, make_loaders
from model import create_model
from train import train_one_epoch, evaluate, DEVICE
from utils import set_seed, compute_metrics

print("GPU:", torch.cuda.is_available(), DEVICE)

EXTRACT_DIR = "/content/ldct_data"
with zipfile.ZipFile("/content/LDCTIQAG2023_train.zip") as z:
    z.extractall(EXTRACT_DIR)
matches = glob.glob(f"{EXTRACT_DIR}/**/train.json", recursive=True)
DATA_DIR = os.path.dirname(matches[0])
print("DATA_DIR:", DATA_DIR, "| görüntü:", len(glob.glob(f"{DATA_DIR}/image/*.tif")))

config = Config(data_dir=Path(DATA_DIR))
df = load_master_list(config.data_dir)
print("Toplam:", len(df))

mse_loss = nn.MSELoss()
OUT_DIR = Path("/content/outputs_b")
OUT_DIR.mkdir(exist_ok=True)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, targets = [], []
    for imgs, labels in loader:
        preds.append(model(imgs.to(DEVICE)).squeeze(1).cpu().numpy())
        targets.append(labels.numpy())
    return np.concatenate(preds), np.concatenate(targets)

In [ ]:
from sklearn.model_selection import KFold

def run_fold(fold, tr_idx, va_idx, num_epochs=config.num_epochs):
    path = OUT_DIR / f"b3_fold{fold}.json"
    save = (num_epochs == config.num_epochs)
    if save and path.exists():
        print(f"[fold {fold}] kayıt var, atlandı")
        return json.loads(path.read_text())
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    va_df = df.iloc[va_idx].reset_index(drop=True)
    set_seed(config.seed)
    train_loader, val_loader, _ = make_loaders(tr_df, va_df, va_df, config)
    set_seed(config.seed)
    model = create_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, mse_loss)
        if epoch % 10 == 0 or epoch == num_epochs:
            val_loss, val_srocc = evaluate(model, val_loader, mse_loss)
            print(f"[fold {fold}] Epoch {epoch:02d}/{num_epochs} | train {train_loss:.4f} | val {val_loss:.4f} | val SROCC {val_srocc:.4f}")
    preds, targets = predict(model, val_loader)
    metrics = compute_metrics(preds, targets)
    result = {"fold": fold, "n_train": len(tr_df), "n_val": len(va_df), "metrics": metrics}
    if save:
        path.write_text(json.dumps(result))
    print(f"[fold {fold}] metrics: {metrics}")
    return result

kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(df))
print([(len(a), len(b)) for a, b in folds])

In [ ]:
cv_results = []
for fold, (tr_idx, va_idx) in enumerate(folds):
    cv_results.append(run_fold(fold, tr_idx, va_idx))

In [ ]:
import pandas as pd

rows = [dict(fold=r["fold"], **r["metrics"]) for r in cv_results]
cv_df = pd.DataFrame(rows).set_index("fold")
cols = ["MAE", "RMSE", "PLCC", "SROCC", "KROCC"]
cv_df[cols].round(4).to_csv(OUT_DIR / "b3_folds.csv")

summary = pd.DataFrame({
    "mean": cv_df[cols].mean(),
    "std": cv_df[cols].std(ddof=1),
}).round(4)
summary.to_csv(OUT_DIR / "b3_summary.csv")

print(cv_df[cols].round(4).to_string())
print()
print(summary.to_string())